# Training a Computer-Using Agent (CUA) for GUI Task Automation
#### Group 5 — CS 1090B

**Members:** William Gao, Jamie Linsdell, Tim Zhang, Zaina Azhar, Guy Mahieu

---

#### Research Question

Can a sub-1B vision-language model (Florence-2, 0.77B), LoRA fine-tuned on UGround-V1-Data,
match or approach a 2B baseline (Qwen2-VL) on UI element grounding accuracy?

#### MS3 Recap

MS3 established zero-shot baselines for Florence-2 and Qwen2-VL on 500 UGround samples.
MS4 adds LoRA fine-tuning for Florence-2 and evaluates against the zero-shot baselines.


## Table of Contents

- [0. Setup](#0-setup)
- [1. Data Loading](#1-data-loading)
- [2. Florence-2 Fine-Tuning](#2-florence-2-fine-tuning)
  - [2.1 Load model + pull training code](#21-load-model--pull-training-code)
  - [2.2 Build train / eval split](#22-build-train--eval-split)
  - [2.3 Smoke test](#23-smoke-test)
  - [2.4 Full training](#24-full-training)
  - [2.5 Evaluate](#25-evaluate)
  - [2.6 Compare: zero-shot vs fine-tuned](#26-compare-zero-shot-vs-fine-tuned)
- [3. Qwen2-VL Fine-Tuning (Tim)](#3-qwen2-vl-fine-tuning-tim)


## 0. Setup

> **transformers version:** Florence-2 requires `==4.44.2`.
> The Qwen section (§3) needs `>=4.45.0` — run the upgrade cell there and restart.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers==4.44.2", "accelerate",
                "peft>=0.12.0", "torchao>=0.16.0"], check=True)
print("Dependencies installed.")


In [ ]:
import os, re, json, io, time, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from datasets import load_dataset
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"Memory : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


## 1. Data Loading

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import glob as _glob

EXTRACT_DIR = "/content/drive/MyDrive/Too CUA for School/uground_subset"
_pq = [f for f in _glob.glob(f"{EXTRACT_DIR}/**/*.parquet", recursive=True)
       if os.path.getsize(f) > 0]
print(f"Found {len(_pq)} parquet files")
ds = load_dataset("parquet", data_files=_pq, split="train")
print(f"Dataset: {len(ds):,} rows")


## 2. Florence-2 Fine-Tuning

### 2.1 Load model + pull training code

In [ ]:
from transformers import AutoProcessor, AutoModelForCausalLM

FLORENCE_MODEL_ID = "multimodalart/Florence-2-large-no-flash-attn"
print(f"Loading {FLORENCE_MODEL_ID}...")
florence_processor = AutoProcessor.from_pretrained(
    FLORENCE_MODEL_ID, trust_remote_code=True)
florence_model = AutoModelForCausalLM.from_pretrained(
    FLORENCE_MODEL_ID, torch_dtype=torch.float16, trust_remote_code=True
).to(DEVICE)
print("Florence-2 loaded.")


In [ ]:
# Pull florence_finetune.py from the MS4 branch
REPO_URL    = "https://github.com/William-Gao/toocuaforschool.git"
REPO_BRANCH = "MS4"
REPO_DIR    = "/content/toocuaforschool"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", REPO_BRANCH], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import importlib
if "florence_finetune" in sys.modules:
    importlib.reload(sys.modules["florence_finetune"])

import florence_finetune as ff
print(f"florence_finetune loaded from: {ff.__file__}")


### 2.2 Build train / eval split

In [ ]:
EVAL_N  = 500
TRAIN_N = 2000   # increase for more thorough training

random.seed(123)
all_idx = list(range(len(ds)))
random.shuffle(all_idx)
EVAL_ROW_SET = set(all_idx[:EVAL_N])

eval_items  = ff.build_training_items(ds, excluded_rows=set(), cap=None, seed=123)
eval_items  = [it for it in eval_items if it["row_idx"] in EVAL_ROW_SET][:EVAL_N]

train_items = ff.build_training_items(ds, excluded_rows=EVAL_ROW_SET, cap=TRAIN_N)
overlap = sum(1 for it in train_items if it["row_idx"] in EVAL_ROW_SET)

print(f"Train: {len(train_items):,} items")
print(f"Eval : {len(eval_items):,} items")
print(f"Overlap: {overlap} (must be 0)")
assert overlap == 0


### 2.3 Smoke test

Run before the full training. Takes ~3-5 min on an A100.
Confirms: loc token vocab, tensor shapes, loss printing, and a quick inference check.


In [ ]:
peft_model = ff.attach_lora(florence_model, r=16, alpha=32, dropout=0.05)

cfg_smoke = ff.TrainCfg(
    output_dir="/content/florence_lora_smoke",
    drive_dir=None,
    smoke=True,
    lambda_soft=0.5,
    sigma=5.0,
    lambda_l2=0.0,
    smoke_max_train_samples=100,
    smoke_max_steps=20,
)
ff.train(peft_model, florence_processor, ds, train_items, cfg_smoke)

smoke_df = ff.evaluate(peft_model, florence_processor, eval_items[:3], ds, verbose_first_n=3)
print(smoke_df[["gt_x", "gt_y", "pred_x", "pred_y", "raw_output"]].to_string())


### 2.4 Full training

Loss = CE + Gaussian Soft-CE (λ=0.5, σ=5.0).
Auto-resumes from Drive checkpoints if the run is interrupted.
Estimated time: ~4-6 hrs on an A100 for 2000 training items.


In [ ]:
DRIVE_BASE = "/content/drive/MyDrive/Too CUA for School/MS4"
os.makedirs(DRIVE_BASE, exist_ok=True)

cfg_full = ff.TrainCfg(
    output_dir="/content/florence_lora_adapter",
    drive_dir=f"{DRIVE_BASE}/florence_lora_adapter",
    smoke=False,
    lambda_soft=0.5,
    sigma=5.0,
    lambda_l2=0.0,
    lr=2e-4,
    batch_size=2,
    grad_accum=4,
    epochs=1,
    save_steps=500,
    logging_steps=50,
    save_total_limit=3,
)
ff.train(peft_model, florence_processor, ds, train_items, cfg_full)


### 2.5 Evaluate

In [ ]:
ft_df = ff.evaluate(peft_model, florence_processor, eval_items, ds)
ft_df.to_csv(f"{DRIVE_BASE}/florence_ft_results.csv", index=False)
print(f"Predictions: {ft_df['pred_x'].notna().sum()}/{len(ft_df)}")


### 2.6 Compare: zero-shot vs fine-tuned

In [ ]:
def compute_metrics(df, label, n_total):
    if df is None:
        return {"Model": label, "Parse rate": "—", "Acc@50px": "—", "Acc@100px": "—", "Mean err (px)": "—"}
    valid = df[df["pred_x"].notna()].copy()
    if len(valid) == 0:
        return {"Model": label, "Parse rate": f"0/{n_total}", "Acc@50px": "0%", "Acc@100px": "0%", "Mean err (px)": "—"}
    px_err = np.sqrt(
        ((valid["pred_x"]/999*valid["width"]) - (valid["gt_x"]/999*valid["width"]))**2 +
        ((valid["pred_y"]/999*valid["height"]) - (valid["gt_y"]/999*valid["height"]))**2
    )
    return {
        "Model":         label,
        "Parse rate":    f"{len(valid)}/{n_total}",
        "Acc@50px":      f"{(px_err<=50).sum()/n_total*100:.1f}%",
        "Acc@100px":     f"{(px_err<=100).sum()/n_total*100:.1f}%",
        "Mean err (px)": f"{px_err.mean():.1f}",
    }

N = len(eval_items)
rows = [
    # Paste in MS3 zero-shot numbers below
    {"Model": "Florence-2 zero-shot (MS3)",  "Parse rate": "—", "Acc@50px": "?", "Acc@100px": "?", "Mean err (px)": "?"},
    {"Model": "Qwen2-VL-2B zero-shot (MS3)", "Parse rate": "—", "Acc@50px": "?", "Acc@100px": "?", "Mean err (px)": "?"},
    compute_metrics(ft_df, "Florence-2 fine-tuned (MS4)", N),
]
print(pd.DataFrame(rows).to_string(index=False))


## 3. Qwen2-VL Fine-Tuning (Tim)

> **Restart required before running this section.**
> Run the cell below, then **Runtime → Restart session**.
> After restarting, run §1 (data loading) then jump here.


In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "transformers>=4.45.0"],
    capture_output=True, text=True
)
print(result.stdout[-300:] if result.stdout else "")
print("\nRestart the runtime now, then run §1 and §3.")
try:
    import google.colab.runtime
    google.colab.runtime.unassign()
except Exception:
    pass
